<div align="left">
    <img src="https://form.ethz.ch/hss26/_jcr_content/par/image/image.imageformat.1286.dpr2.1527502072.png" alt="Earth Observation Foundation Models Workspace" width="70%" style="max-height: 350px; object-fit: cover; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
</div>

# Hands-on Session: Earth Observation (EO) Foundation Models
## 🛰️ Advanced Geospatial Representation
### 🔍 Technical Workflow: Framework Execution

**Case study:** Trentino region — Val di Fiemme forest, devastated by the **Vaia storm (October 2018)**.  
We use this event as a ground-truth anchor to validate whether the pipeline can autonomously detect the abrupt landscape change without any labelled data.

**Pipeline steps:**

1. **Load & process HLS data from GEE**
   - Query the Harmonized Landsat Sentinel-2 (HLS) collection from Google Earth Engine
   - Filter by area of interest (Val di Fiemme) and time range spanning pre- and post-Vaia (2017–2019)

2. **Load the foundation model**
   - Initialize pre-trained weights of **Clay v1.5** via the `terratorch` registry (`timm_clay_v1_base`)

3. **Generate high-dimensional embeddings**
   - Pass image tensors through the Vision Transformer (ViT) encoder
   - Each observation date is mapped to a dense feature vector — a mathematical "fingerprint" of the landscape at that moment

4. **Feature reduction & visualization (PCA & t-SNE)**
   - **PCA** isolates the directions of maximum variance across dates, surfacing the most structurally significant changes
   - **t-SNE** projects the high-dimensional space into 2D, making temporal clusters and outliers (e.g. the post-Vaia observations) visually separable

5. **Change date detection**
   - Compare feature spaces across time steps
   - Identify the date where the embedding distribution shifts most abruptly — expected to align with **October 2018**

---

### 💡 Core Takeaways

#### 1. HLS data
- A radiometrically harmonized, analysis-ready product fusing Landsat and Sentinel-2
- Provides consistent multi-spectral time series — essential for temporal change analysis

#### 2. Geospatial foundation models
- **Multi-spectral design:** ingest any band combination (RGB, NIR, SWIR) in any order — not limited to 3-channel RGB
- **Self-supervised pre-training:** trained on millions of unlabelled global images via Masked Autoencoders (MAE), learning Earth surface physics with no manual annotation
- **Zero-shot capability:** no fine-tuning needed for downstream tasks like change detection

#### 3. Embeddings as landscape fingerprints
- Instead of pixel-by-pixel band comparisons, the model encodes the full structural pattern, canopy texture, and moisture status into a single vector
- Changes in the embedding space directly reflect real-world landscape changes — in our case, the mass windthrow caused by Vaia across Val di Fiemme.

## Setup
Install required packages

In [ ]:
# Install packages
!pip install geemap
!pip install --upgrade -q pip
!pip install -q opencv-python terratorch>=1.1.0 einops scikit-learn>=1.5.0

In [ ]:
import ee
import geemap

import cv2
import torch
import numpy as np
import time
from einops import rearrange, reduce
from matplotlib import pyplot as plt
from sklearn import decomposition, preprocessing
from terratorch import BACKBONE_REGISTRY
from sklearn.manifold import TSNE
import torchvision.transforms.v2 as v2
import yaml
from box import Box
from datetime import datetime


# Earth Engine Tutorial: Harmonized Landsat Sentinel-2 (HLS) Data Pipeline

This notebook guides you step-by-step through filtering, subsetting, visualizing, and batch-exporting HLS imagery over a specific Region of Interest (ROI) before and after a specific historical cutoff date.

### Prerequisites
Make sure you have access to a **Google Earth Engine (GEE)** account. When running the initialization cell, you will be prompted to authenticate through your browser.

### Step 1: Initialize GEE
We need `geemap` for interactive map visualizations natively within the Colab environment, and the `ee` library to communicate with the Google Earth Engine API.

### Google Earth Engine requires a Google Cloud Project ID to run.
#
### How to find your Project ID:
1. Open a new tab and go to: https://code.earthengine.google.com/
2. Look at the top-right corner next to your profile picture/name.
3. Copy the project text ID shown there (e.g., 'ee-yourusername' or 'my-project-1234').
4. Paste that text string inside the single quotes for PROJECT_ID below.

In [ ]:
# 1. Authenticate to Earth Engine
ee.Authenticate()

# 3. Initialize with the project ID
ee.Initialize(project="km-project-01-497412")

### Step 2: Define Region of Interest (ROI)
We define a point using a specific latitude and longitude (located in val di fiemme), create a buffer of 3,840 meters around it, and find its bounding box coordinates (`bounds()`) to use as our target window.

In [ ]:
lat = 46.231128
lon = 11.407454

ROI = ee.Geometry.Point([lon, lat]).buffer(3840).bounds()

print("ROI defined successfully.")

### Step 3: Load and Filter Base Image Collection
We target the **NASA/HLS/HLSS30/v002** image collection (Harmonized Landsat Sentinel-2). To ensure we don't handle empty images or data fragments, we write a mapping function to count valid pixels inside our ROI using `reduceRegion` and filter out any completely blank scenes.

In [ ]:
# Load the raw collection overlapping our ROI
collection = ee.ImageCollection('NASA/HLS/HLSS30/v002').filterBounds(ROI)

# Define helper mapping function to get valid pixel counts inside our ROI boundary
def count_pixels(img):
    count = img.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=ROI,
        scale=30,
        maxPixels=1e13
    ).values().get(0)
    return img.set('pixel_count', count)

# Map the function and filter collection
validCollection = collection.map(count_pixels).filter(ee.Filter.gt('pixel_count', 0))

print("Collection filtered for valid pixels successfully.")

### Step 4: Image Selection Strategy (2018 vs 2019)
We select imagery from two fixed yearly windows: **2018** (pre-storm) and **2019** (post-storm). We filter out heavily cloudy scenes (keeping less than 30% cloud cover), sort by clarity, and select the **20 images from each year**, giving a final collection of 40 images.

In [ ]:
# Filter 20 clearest images from 2018 (pre-Vaia)
beforePool = (validCollection
              .filterDate('2018-01-01', '2018-12-31')
              .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
              .sort('CLOUD_COVERAGE'))

# Filter 20 clearest images from 2019 (post-Vaia)
afterPool = (validCollection
             .filterDate('2019-01-01', '2019-12-31')
             .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
             .sort('CLOUD_COVERAGE'))

# Select top 20 clearest from each year and merge
before = beforePool.limit(20)
after = afterPool.limit(20)
selected = before.merge(after)

print(f"2018 images:  {before.size().getInfo()}")
print(f"2019 images: {after.size().getInfo()}")
print(f"Total combined images:   {selected.size().getInfo()}")

### Step 5: Extracting Metadata & Dates to Client
To make working with the images easier, we'll extract the system timestamp and parse it into a clean readable string format (`YYYY-MM-dd`). Then we will pull this date array into python's client side so you can easily choose which one to preview.

In [ ]:
# Map function to add readable text dates as metadata attribute
def add_date_str(img):
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return img.set('date_str', date)

withDate = selected.map(add_date_str)

# Convert Server-side Collection to Server List and fetch array attributes to client
list_images = withDate.toList(40)
dates = withDate.aggregate_array('date_str').getInfo()

print("Available Scene Dates mapped to indices:")
for idx, date_str in enumerate(dates):
    #period = "BEFORE cutoff" if idx < 15 else "AFTER cutoff"
    print(f"Index [{idx}]: {date_str}")

### Step 5: Visual Inspection & Scene Selection

Before passing images into the model, we visually inspect each scene to hand-pick the **6 cleanest images per year** — free of snow and cloud cover.

Change `viewIndex` (0–39) to browse through all 40 images:
- **0–19** → 2018 scenes
- **20–39** → 2019 scenes

Scenes to **exclude**:
1. Heavy cloud cover obscuring the forest canopy
2. Snow cover altering the spectral signature

Note the indices of the 6 best scenes per year before proceeding to the next step.

In [ ]:
# CHANGE THIS INDEX VALUE TO TEST DIFFERENT TIMESTAMPS
viewIndex = 39

# Fetch the specified target image
imageToVisualize = ee.Image(list_images.get(viewIndex)).clip(ROI)
imageDateStr = dates[viewIndex]

# Build interactive map workspace
Map = geemap.Map()
Map.centerObject(ROI, 12)

# Configure True Color RGB Render parameter values
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.005,
    'max': 0.2,
    'gamma': 1.9
}

Map.addLayer(imageToVisualize, vis_params, f'HLS RGB Scene [{viewIndex}]: {imageDateStr}')
print(f"Displaying Scene captured on: {imageDateStr}")
Map


### STEP 6b:  SELECTION (6 images from 2018 & 6 images from 2019)

### INSTRUCTIONS:

Based on your visual inspection using the map widget above, choose:


1.   Less cloudy 6 image indices from 2018 (Indices 0 to 19)
2.   Less cloudy 6 image indices from 2019 (Indices 20 to 39).

Replace the numbers in the lists below with your chosen index choices.


In [ ]:
# Example selection: Enter your chosen indices here
chosen_2018_indices = [0, 2, 6, 7, 10, 11]     # Must be indices between 0 and 19
chosen_2019_indices  = [22, 23, 24, 27, 33, 38] # Must be indices between 20 and 39

# Combine the choices into a single index list
final_indices = chosen_2018_indices + chosen_2019_indices

# Basic validation check for the students
if len(chosen_2018_indices) != 6 or len(chosen_2019_indices) != 6:
    print("[WARNING]: Please ensure you select exactly 6 images for both pools!")

# Extract the chosen images from our master list using Earth Engine server logic
final_image_list = []
final_dates_list = []

print("Extracting your final curated dataset collection...")
for index in final_indices:
    final_image_list.append(ee.Image(list_images.get(index)))
    final_dates_list.append(dates[index])
    print(f"-> Retained Index [{index}]: Captured on {dates[index]}")

# Convert our selected list references back into a workable Server List object
# so the down-stream export code works perfectly
list_images = ee.List(final_image_list)
dates = final_dates_list

print("\n[SUCCESS]: Final 12 scenes locked in. Proceed to the batch export cell below.")

### Step 7: Batch Export to Google Drive

Before running the foundation model locally, we export the selected scenes from GEE to Google Drive as analysis-ready image chips.

**What gets exported:**
- **Bands:** `B2, B3, B4, B8A, B11, B12` — the exact 6-band combination expected by Clay (Blue, Green, Red, Narrow NIR, SWIR-1, SWIR-2)
- **Size:** 256×256 pixels per chip — the native patch size of the ViT encoder
- **Projection:** EPSG:32632 (UTM Zone 32N — appropriate for the Val di Fiemme area)
- **Format:** Float32, one file per date, named `HLS_Clay_YYYY-MM-DD`

One export task is submitted per image (12 total). Tasks run asynchronously in the GEE backend — monitor progress in the **GEE Task Manager** before moving to the next step.

In [ ]:
for i in range(12):
    img = ee.Image(list_images.get(i))
    dateStr = dates[i]

    # Filter down specific channel combinations for Machine Learning
    mlInput = img.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']).toFloat()

    # Setup Drive Export Pipeline profile parameter options
    task = ee.batch.Export.image.toDrive(
        image=mlInput.clip(ROI),
        description=f'HLS_Clay_{dateStr}',
        fileNamePrefix=f'HLS_Clay_{dateStr}',
        region=ROI,
        dimensions=(256, 256),
        crs='EPSG:32632',
        maxPixels=1e13
    )

    # Fire processing request parameters straight into asynchronous Engine worker queues
    task.start()
    print(f"Submitted batch processing pipeline job: HLS_Clay_{dateStr}")

print("\nAll 12 background engine threads pushed to tracking workspace successfully! Open your GEE dashboard or task runner to review file rendering updates.")

# Section 2: Generating Deep Geospatial Embeddings with Clay

Now that we have selected and verified our curated sequence of clean HLS images, we will leverage **Clay**—a state-of-the-art spatial foundation model designed for multi-spectral remote sensing analysis.

Unlike Prithvi (which uses a 3D spatio-temporal architecture requiring fixed sequence windows of 4 dates), Clay is built on a 2D Vision Transformer (ViT) architecture that handles standalone single-date observation captures natively. It processes individual spatial scenes into high-dimensional feature representations called **embeddings** that preserve fine-grained localized land disturbances, vegetation shifts, and canopy state conditions without mixing temporal inputs.

### Step 8: Initialize Modeling Runtime Environment
We check for hardware device acceleration (CUDA GPU) and ensure the necessary geospatial model modules are ready for execution.

In [ ]:
# Verify if GPU acceleration is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware device acceleration: {device}")
print(f"Active CV2 Version: {cv2.__version__}")

###Step 9: Discover the Exact Registry Name
Run this quick snippet in a temporary cell to inspect the actual registry strings:

In [ ]:
# Print out all keys inside the registry that contain the word 'clay'
clay_models = [m for m in BACKBONE_REGISTRY if "clay" in m.lower()]
print("Available Clay Backbones in your version of terratorch:")
print(clay_models)

###Step 2: The Quick Clean Fix
Once you see the exact string printed by Step 1, update model_name_target to match it. For example, if it says "clay_v1", your loading block will look like this:

In [ ]:
# 1. Update this variable with the exact name found from the step above
model_name_target = "timm_clay_v1_base"  # Or whatever string printed above!

# 2. Check registry safely
assert any(model_name_target in m for m in BACKBONE_REGISTRY), f"Target model backbone {model_name_target} not found!"

print(f"--> Downloading and instantiating Clay model weights ({model_name_target}) from Hugging Face...")

# 3. Build the model architecture natively through the registry
model = BACKBONE_REGISTRY.build(model_name_target, pretrained=True)
model.to(device)
model.eval()

print(f"\n[SUCCESS]: {model_name_target} successfully loaded and allocated to memory.")

### Step 10: Data Preparation & Normalization

Before feeding the images into Clay, we prepare the raw GEE data into model-ready tensors in three steps:

1. **Sort chronologically** — images are ordered by acquisition date to ensure the temporal sequence is correct

2. **Normalize with HLS statistics** — each band is normalized using HLS-specific mean and standard deviation values (pre-computed from global HLS data), ensuring the input distribution matches what Clay was trained on

3. **Crop & reformat each scene:**
   - Downloaded at native **30 m resolution** from GEE
   - Center-cropped to exactly **256×256 pixels** (no rescaling or blurring)
   - Rescaled from integer reflectance (0–10000) to float (0.0–1.0)
   - Transposed from HWC → CHW format as expected by PyTorch
   - Stacked into a single array of shape **(12, 6, 256, 256)** — 12 dates × 6 bands × 256 × 256 pixels

In [ ]:
print("1. Enforcing chronological order on selected images...")
temp_collection = ee.ImageCollection(list_images).sort('system:time_start')
sorted_list_images = temp_collection.toList(12)
sorted_dates = temp_collection.aggregate_array('date_str').getInfo()

print("\n2. Initializing HLS spectral calibration parameters...")
hls_band_order = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']
hls_means = [0.0498, 0.0510, 0.0543, 0.2238, 0.1747, 0.0963]
hls_stds  = [0.0381, 0.0396, 0.0470, 0.1171, 0.0991, 0.0768]

transform = v2.Compose([
    v2.Normalize(mean=hls_means, std=hls_stds),
])

print("\n3. Fetching native HLS matrices and enforcing exact 256x256 cropping limits...")
image_arrays = []
for i in range(12):
    img_ee = ee.Image(sorted_list_images.get(i))
    mlInput = img_ee.select(hls_band_order)

    # Download raw pixels from GEE at native 30-meter scale resolution
    img_np = geemap.ee_to_numpy(mlInput, region=ROI, scale=30) # e.g., returns (265, 264, 6)

    # --- NATIVE BOUNDARY CROPPING (No blurring or spatial stretching) ---
    h, w, c = img_np.shape

    # Calculate starting indices to slice out an exact 256x256 block from the center
    start_h = (h - 256) // 2
    start_w = (w - 256) // 2

    # Slice the array natively down to exactly (256, 256, 6)
    img_np_cropped = img_np[start_h:start_h+256, start_w:start_w+256, :]

    # Scale HLS surface reflectance integers (0-10000) down to raw decimals (0.0-1.0)
    img_np_cropped = img_np_cropped.astype(np.float32) / 10000.0

    # Transpose layout from HWC (256, 256, 6) -> CHW (6, 256, 256)
    img_np_final = np.transpose(img_np_cropped, (2, 0, 1))

    # Process through our static normalization transform
    img_tensor = torch.from_numpy(img_np_final).float()
    normalized_tensor = transform(img_tensor)

    image_arrays.append(normalized_tensor.numpy())

full_stack = np.stack(image_arrays, axis=0)
print(f"\n[SUCCESS]: Generated {len(full_stack)} raw, normalized satellite maps.")
print(f"Final Data Matrix Array Layout Dimensions: {full_stack.shape}")

### Step 11: Pass Standalone Images Through Clay to Extract Latent Embeddings
We execute forward passes across all 12 dates. To capture fine-grained localized spatial details across the canopy, we pull the un-averaged high-dimensional feature blocks from Clay and flatten them into a dense representation matrix.

In [ ]:
print("1. Executing forward pass natively via Terratorch Backbone Wrapper...")
individual_embeddings = []
start_time = time.time()

clay_band_names = ['blue', 'green', 'red', 'nir', 'swir16', 'swir22']

for i in range(12):
    single_image = full_stack[i] # Layout Shape: (6, 256, 256)
    input_tensor = torch.from_numpy(single_image).float().unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            features = model(input_tensor, bands=clay_band_names)
        except TypeError:
            features = model(input_tensor)

        # Unpack output representations safely from the wrapper structure
        if isinstance(features, list):
            feature_tensor = features[-1]
        elif isinstance(features, dict):
            feature_tensor = features[list(features.keys())[0]]
        else:
            feature_tensor = features

        # Flatten spatial tokens into a single sequence vector row
        flat_vector = feature_tensor.detach().cpu().numpy().flatten()
        individual_embeddings.append(flat_vector.tolist())
        print(f"   └─► Day {i:02d} ({sorted_dates[i]}) features compiled successfully.")

# Stack entries into your final 2D matrix for Step 12 (PCA / t-SNE)
individual_matrix = np.vstack([np.array(x) for x in individual_embeddings])

print(f"\n[SUCCESS]: Feature extraction complete!")
print(f"Total processing time: {time.time() - start_time:.2f} seconds.")
print(f"Dense Feature Matrix Layout Dimensions: {individual_matrix.shape}")

### Step 12: High-Dimensional Variance Mapping & Neighborhood Clustering (PCA vs t-SNE)

#### **The Technical Objective**
Now that we have high-dimensional features for each day, we use dimensionality reduction to map the land's trajectory. Since remote sensing data captures overlapping environmental variables (weather patterns vs. physical ground modifications), we pass our features through two distinct reduction approaches:

1. **Z-Score Feature Standardization (`StandardScaler`)**: Normalizes feature channels to ensure that transient, high-magnitude pixel variations do not mathematically overpower the global variance landscape.
2. **Linear Orthogonal Projections (PCA)**: Extracts the primary axes of variance ($PC_1$ vs $PC_2$). This decouples continuous atmospheric shifts from true physical changes on the ground (such as canopy clearing).
3. **Non-Linear Proximity Mapping (t-SNE)**: Fits the high-dimensional feature coordinates onto a low-dimensional manifold. By looking at geometric proximities instead of global variance lines, t-SNE groups highly-altered post-event days into isolated semantic neighbor clusters.

*Note: Since our dataset contains 12 standalone dates, we explicitly set `perplexity=3` to perform accurate neighborhood probability calculations safely without exceeding the sample size limits.*

In [ ]:
print("1. Normalizing individual feature channels...")
scaler = preprocessing.StandardScaler()
scaled_matrix = scaler.fit_transform(individual_matrix)


print("2. Projecting linear variance along orthogonal components...")
pca_individual = decomposition.PCA(n_components=2)
pca_results = pca_individual.fit_transform(scaled_matrix)

var1 = pca_individual.explained_variance_ratio_[0] * 100
var2 = pca_individual.explained_variance_ratio_[1] * 100


print("3. Computing non-linear proximity manifolds via t-SNE...")
# Adaptively set perplexity safely below sample count constraints
safe_perplexity = min(3, len(individual_matrix) - 1)

tsne = TSNE(n_components=2, random_state=42, perplexity=safe_perplexity, max_iter=1000)
tsne_results = tsne.fit_transform(scaled_matrix)
print("[SUCCESS]: Dimensionality reduction layers completed.")


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(26, 7))
timeline_colors = plt.cm.coolwarm(np.linspace(0, 1, 12))

# --- Panel 1: PCA Component 1 (Atmospheric Track) ---
for i in range(12):
    ax1.scatter(i, pca_results[i, 0], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax1.plot(range(12), pca_results[:, 0], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xticks(range(12))
ax1.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax1.set_xlabel("Exact Satellite Capture Date")
ax1.set_ylabel("PCA Component 1 Coordinate")
ax1.set_title(f"PCA Component 1 ({var1:.1f}% Variance)")
ax1.grid(True, alpha=0.2)

# --- Panel 2: PCA Component 2 (Canopy/Landscape Track) ---
for i in range(12):
    ax2.scatter(i, pca_results[i, 1], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax2.plot(range(12), pca_results[:, 1], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(12))
ax2.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax2.set_xlabel("Exact Satellite Capture Date")
ax2.set_ylabel("PCA Component 2 Coordinate")
ax2.set_title(f"PCA Component 2 ({var2:.1f}% Variance)")
ax2.grid(True, alpha=0.2)

# --- Panel 3: t-SNE Non-Linear Neighborhood Groupings ---
for i in range(12):
    label_text = f"Day {i:02d}: {sorted_dates[i]}"
    ax3.scatter(
        tsne_results[i, 0],
        tsne_results[i, 1],
        color=timeline_colors[i],
        s=300,
        edgecolors='black',
        label=label_text,
        zorder=3
    )
ax3.set_xlabel("t-SNE Dimension 1")
ax3.set_ylabel("t-SNE Dimension 2")
ax3.set_title("t-SNE Semantic Manifold Clusters")
ax3.grid(True, alpha=0.2)

# Position chronological legends cleanly outside the figure space
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Exact Capture Timeline")
plt.tight_layout()
plt.show()

print("Execution complete! Compare the linear separation in Panels 1 & 2 against the non-linear semantic neighborhood clusters in Panel 3.")


### Exercise 1: Random Forest Classification on Embeddings

Instead of just visualising the feature space, can we **classify** each scene as pre- or post-Vaia using the embeddings as input features?

**How to do it:**
- Use `full_stack` (shape `12 × 6 × 256 × 256`) — flatten each scene's embedding into a 1D feature vector
- Assign binary labels: `0` for 2018 scenes, `1` for 2019 scenes
- Train a `RandomForestClassifier` from scikit-learn using a leave-one-out or train/test split

**What to expect:**
- High accuracy (ideally 100%) would confirm the embeddings carry enough discriminative information to separate pre- from post-storm scenes
- Low accuracy suggests the selected scenes are too noisy or too few — a good reason to go back and refine your scene selection

> 💡 *Hint: try adding the noisy scenes from Exercise 1 to your training set — does more data help or hurt the classifier?*

In [ ]:
#Sample code
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

# Flatten embeddings: (12, N) feature matrix
X = full_stack.reshape(12, -1)
y = [0] * 6 + [1] * 6  # 6 pre-Vaia, 6 post-Vaia

# Leave-one-out cross-validation (suitable for small sample sizes)
loo = LeaveOneOut()
preds = []
for train_idx, test_idx in loo.split(X):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X[train_idx], np.array(y)[train_idx])
    preds.append(clf.predict(X[test_idx])[0])

print(f"LOO Accuracy: {accuracy_score(y, preds) * 100:.1f}%")


### Exercise 2: What Happens With Noisy Scenes?

Re-run the pipeline including **cloudy or snowy images** you excluded in Step 5 and observe how the embedding space reacts.

**How to do it:**
- Manually add back the snowy/cloudy scene indices you noted during visual inspection
- Re-run later steps with the new image set

**What to expect:**
- Cloudy scenes will likely appear as **outliers** in both PCA and t-SNE — cloud cover dominates the spectral signal and pushes the embedding far from clear-sky observations
- Snowy scenes will form their **own cluster**, since snow has a very distinct spectral signature (high reflectance across visible bands, low in SWIR)


> 💡 *Hint: if the 2018 vs 2019 separation disappears after adding noisy images, try normalizing or removing the outliers and re-running — what changes?*